In [1]:
import pandas as pd
import geopandas as gpd
import altair as alt
import json
from shapely.geometry import shape, Polygon
from shapely import wkt
from shapely.ops import linemerge, unary_union
alt.data_transformers.disable_max_rows()
import h3

In [8]:
with open('Data/Metro/metro_trajectories.json') as f:
    trajectories = json.load(f)


df_trajectories = []
for trajectory in trajectories['features']:
    properties = trajectory.get('properties', {})
    tram = properties.get('NOM_TRAM_LINIA')
    nom_linia = properties.get('NOM_LINIA')
    origen = properties.get('CODI_ESTACIO_INI')
    dest = properties.get('CODI_ESTACIO_FI')                   
    df_trajectories.append({
            'origen': origen,
            'dest': dest,
            'tram': tram,
            'linia': nom_linia,
            'type': 'Metro',
            'geometry': shape(trajectory['geometry'])
        })
df_trajectories_all = pd.DataFrame(df_trajectories)
geo_df_trajectories_all = gpd.GeoDataFrame(df_trajectories_all, geometry='geometry', crs=4326)
geo_df_trajectories_all = geo_df_trajectories_all[geo_df_trajectories_all['tram'].str.contains('Inici') == False]
geo_df_trajectories_all = geo_df_trajectories_all[geo_df_trajectories_all['tram'].str.contains('Final') == False]
geo_df_trajectories_all.to_crs('EPSG:25831', inplace=True)
geo_df_trajectories_all['length'] = geo_df_trajectories_all['geometry'].length 
geo_df_trajectories_all['speed'] = 25 /3.6 # 25 kmh to ms like in the paper
geo_df_trajectories_all['time'] = (geo_df_trajectories_all['length'] / geo_df_trajectories_all['speed']) / 60
# geo_df_trajectories_all['time'] = pd.to_timedelta(
#     geo_df_trajectories_all['length'] / geo_df_trajectories_all['speed'],
#     unit='s'
# )
# geo_df_trajectories_all['time'] = geo_df_trajectories_all['time'].apply(
#     lambda x: f"{int(x.total_seconds() // 60):02d}:{int(x.total_seconds() % 60):02d}"
# )
geo_df_trajectories_all['directed'] = False
geo_df_trajectories_all.to_crs('EPSG:4326', inplace=True)
geo_df_trajectories_all['directed'] = False
geo_df_trajectories_all = geo_df_trajectories_all[['origen', 'dest', 'tram', 'linia', 'type', 'length', 'speed', 'time', 'directed','geometry']]
geo_df_trajectories_all['origen'] = 'M' + '-' + geo_df_trajectories_all['linia'] + '-' + geo_df_trajectories_all['origen'].astype(str)
geo_df_trajectories_all['dest'] = 'M' + '-' + geo_df_trajectories_all['linia'] + '-' + geo_df_trajectories_all['dest'].astype(str)
geo_df_trajectories_all.to_crs('EPSG:4326', inplace=True)
excluded = ['L9N','L9S','L10N','L10S','L11','FM','TM']
geo_df_trajectories_all = geo_df_trajectories_all[~geo_df_trajectories_all['linia'].isin(excluded)]

In [9]:
geo_df_trajectories_all

,origen,dest,tram,linia,type,length,speed,time,directed,geometry
1,M-L1-111,M-L1-112,Hospital de Bellvitge - Bellvitge,L1,Metro,882.209743,6.944444,2.117303,False,"MULTILINESTRING ((2.10724 41.34468, 2.10773 41..."
2,M-L1-112,M-L1-113,Bellvitge - Av. Carrilet,L1,Metro,1133.654863,6.944444,2.720772,False,"MULTILINESTRING ((2.11092 41.35098, 2.11052 41..."
3,M-L1-113,M-L1-114,Av. Carrilet - Rambla Just Oliveras,L1,Metro,660.855917,6.944444,1.586054,False,"MULTILINESTRING ((2.10263 41.35855, 2.10202 41..."
4,M-L1-114,M-L1-115,Rambla Just Oliveras - Can Serra,L1,Metro,572.330154,6.944444,1.373592,False,"MULTILINESTRING ((2.09975 41.36409, 2.09946 41..."
5,M-L1-115,M-L1-116,Can Serra - Florida,L1,Metro,616.671198,6.944444,1.480011,False,"MULTILINESTRING ((2.10276 41.36769, 2.10381 41..."
...,...,...,...,...,...,...,...,...,...,...
122,M-L5-529,M-L5-530,Virrei Amat - Vilapicina,L5,Metro,636.238350,6.944444,1.526972,False,"MULTILINESTRING ((2.17491 41.4297, 2.17486 41...."
123,M-L5-530,M-L5-531,Vilapicina - Horta,L5,Metro,683.078591,6.944444,1.639389,False,"MULTILINESTRING ((2.16762 41.43047, 2.16564 41..."
124,M-L5-531,M-L5-532,Horta - El Carmel,L5,Metro,851.210213,6.944444,2.042905,False,"MULTILINESTRING ((2.15997 41.42969, 2.15974 41..."
125,M-L5-532,M-L5-533,El Carmel - El Coll | La Teixonera,L5,Metro,838.886751,6.944444,2.013328,False,"MULTILINESTRING ((2.1551 41.42445, 2.15513 41...."


In [10]:
geo_df_trajectories_all.to_csv("Edges/Metro.csv", index=False)